In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
from PIL import Image
import os
import shutil

# 1. Start with the base model
model = models.resnet18(weights=None) # No need to download ImageNet weights again

# 2. Modify the head exactly like before (during training)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2) 

# 3. Setup the device (M2)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

In [ ]:


# 4. Load the saved state dictionary
model_path = 'tripos_classifier_v4.pth'
model.load_state_dict(torch.load(model_path, map_location=device))

# 5. Set to Evaluation Mode
model.eval() 
print("Model loaded")



Model loaded and ready for inference!


In [ ]:

# 2. Setup paths
unlabeled_folder = '/Users/pao/Documents/CS199/CS199_repository/image_extraction/output_pngs/D20250616T050815_IFCB202'
destination_folder = './potential_tripos_hits_v3_2'
os.makedirs(destination_folder, exist_ok=True)

# 3. Prediction Loop
# Use the same transforms as your validation set
inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Scanning images...")
for img_name in os.listdir(unlabeled_folder):
    if img_name.endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(unlabeled_folder, img_name)
        img = Image.open(img_path).convert('RGB')
        
        # Prepare image for model
        input_tensor = inference_transforms(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(input_tensor)
            # Convert raw output to probabilities (0.0 to 1.0)
            probabilities = torch.nn.functional.softmax(output[0], dim=0)
            
            # 'tripos' is index 1
            tripos_conf = probabilities[1].item() 
            
            if tripos_conf > 0.75:  # Adjust this threshold based on your needs
                shutil.copy(img_path, os.path.join(destination_folder, img_name))
                print(f"Found potential Tripos: {img_name} ({tripos_conf:.2%})")

print("Cleaning complete!")

Scanning images...
Found potential Tripos: 5686.png (84.36%)
Found potential Tripos: 1619.png (75.24%)
Found potential Tripos: 6175.png (80.55%)
Found potential Tripos: 5118.png (78.00%)
Found potential Tripos: 6001.png (80.60%)
Found potential Tripos: 2879.png (75.62%)
Found potential Tripos: 1709.png (78.19%)
Found potential Tripos: 316.png (81.10%)
Found potential Tripos: 3257.png (78.57%)
Found potential Tripos: 884.png (89.95%)
Found potential Tripos: 4703.png (76.57%)
Found potential Tripos: 255.png (79.99%)
Found potential Tripos: 1850.png (76.53%)
Found potential Tripos: 522.png (82.58%)
Found potential Tripos: 2743.png (79.70%)
Found potential Tripos: 4252.png (76.74%)
Found potential Tripos: 2219.png (82.77%)
Found potential Tripos: 6043.png (75.03%)
Found potential Tripos: 1301.png (91.50%)
Found potential Tripos: 3066.png (77.37%)
Found potential Tripos: 1039.png (81.67%)
Found potential Tripos: 2646.png (93.22%)
Found potential Tripos: 2320.png (86.04%)
Found potential Tri